# Mini Project 1 — CourtListener Citation Analysis

**Name:** Ranjitha Rangaswamy  
**Dataset:** `data/courtlistener.csv` (CourtListener API export)  
**Date:** 15 May 2026

This folder is a **standalone** mini project: data, charts, and notebook live in `MiniProject1/` so reviewers can open one directory on GitHub without hunting other weeks.


In [1]:
# Setup — run this cell first (required)
!pip install jupyter plotly kaleido pandas dash dash-mantine-components

import subprocess
import sys
from pathlib import Path

# Fallback if shell `pip` is not on PATH (common in some Jupyter shells)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "jupyter",
        "plotly",
        "kaleido",
        "pandas",
        "dash",
        "dash-mantine-components",
        "-q",
    ],
    stdout=subprocess.DEVNULL,
)

import pandas as pd

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data" / "courtlistener.csv").exists():
    alt = PROJECT_DIR / "MiniProject1"
    if (alt / "data" / "courtlistener.csv").exists():
        PROJECT_DIR = alt
        sys.path.insert(0, str(PROJECT_DIR))

sys.path.insert(0, str(PROJECT_DIR))
import mp1_charts as charts

DATA_CSV = PROJECT_DIR / "data" / "courtlistener.csv"
IMAGES_DIR = PROJECT_DIR / "images"
print("Project dir:", PROJECT_DIR.resolve())
print("Data file exists:", DATA_CSV.exists())


zsh:1: command not found: pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


Project dir: /Users/rnr6/Documents/HCDE/hcde530/MiniProject1
Data file exists: True


---

## Section 1 — Overview

### What is this dataset?

This analysis uses **CourtListener** — a free legal research platform — exported to `data/courtlistener.csv` (~**39,000 rows**). Each row is one opinion with `case` (title), `judge`, `court` / `court_id`, `date_of_decision`, and `cite_count` (incoming citation clusters in this extract).

The file combines two Seattle-area searches:

- **Federal:** U.S. District Court, Western District of Washington (`wawd`) — ~3,000 rows; **non-zero** `cite_count` in this pull.
- **State:** Washington Court of Appeals (`washctapp`) — ~36,000 rows; here **`cite_count` is 0 for every row** and `judge` is mostly missing.

**Source:** [CourtListener API](https://www.courtlistener.com/help/api/) search/export, stacked in earlier course weeks and copied into this folder for MP1.

### What do these terms mean?

Plain-language glossary for readers who are not lawyers or data engineers.

| Term | In plain English |
|------|------------------|
| **CourtListener** | A free public website and database of court opinions, similar in spirit to a searchable library of legal decisions. |
| **Opinion / `case` (title)** | One written court decision. In this file, `case` is the case name (caption), e.g. *Smith v. Jones*. |
| **Citation / `cite_count`** | How often other cases reference this opinion — a rough “how much other courts build on this decision” signal in this export. Higher usually means the opinion is treated as more influential in the data provider’s count. |
| **W.D. Wash. / `wawd`** | **W**estern **D**istrict of **Wash**ington — a **federal** trial-level court covering western Washington (not the state supreme court). Short code in the data: `wawd`. |
| **`washctapp`** | **Wash**ington **Ct**. **App**. — the **state** Court of Appeals (appellate level below the state supreme court). Short code in the data: `washctapp`. |
| **King County pull** | Opinions from a CourtListener search scoped to King County / Seattle-area matters on the state side of the dataset. |
| **Federal vs state court** | Federal courts handle federal law cases; state courts handle state law cases. This project mixes both in one file, so metrics must be read **by court**, not blended. |
| **`court_id`** | Machine-friendly court label (`wawd` or `washctapp`) used in filters and charts. |
| **`court_level`** | Readable label derived from `court_id` (federal district vs state appellate). |
| **Jurisdiction** | Which court’s geographic and legal territory you are looking at — here, mainly W.D. Wash. (`wawd`) for “most cited in this federal district.” |
| **Judge / `judge_clean`** | The authoring judge when the source lists one. Mostly filled in for federal rows; usually blank on state rows in this pull. |
| **Citator** | A professional tool lawyers use to trace who cited whom. This class project is **exploratory**, not a replacement for those tools. |
| **Row** | One line in the spreadsheet — here, one opinion record (sometimes duplicated captions appear on multiple rows). |

### Why these questions matter

I want to see **which opinions carry citation signal in public data** and whether that differs by court level, which is relevant for legal reference workflows and for HCD practice (user centered data interpretation for legal workflows, where labels exist for which population a metric describes).

### Three analytical questions (MP1a)

1. Which judges author the most cited opinions — and does citation frequency vary by **court level**? *(Shown as mean cites by month and court; judges are only reliable on the federal slice.)*
2. What are the **most cited judgments** in a specific jurisdiction? *(Western District of Washington.)*
3. What is the **average** cite behavior per opinion row, and which **case titles** accumulate the most citation mass?

**Practical use:** A lawyer can scan dominant cited titles in this territory; the analysis also warns when state and federal rows **cannot** be compared on `cite_count` without relabeling.

**Week 6 vs this folder:** `week 6/Week6_files/week6_mini_analytics.ipynb` only asks **mini** questions (e.g. how many rows fall in 2025, federal vs state counts). Those three bullets above are **only** developed here in Mini Project 1.


---

## Section 2 — Data Profile

Load the dataset and answer:

- How many rows and columns?
- What does each column represent?
- Any data quality issues (missing values, types, inconsistent formatting)?
- Which columns will the analysis focus on, and why?

Run four first-look operations below; the interpretation cell answers these bullets.


In [2]:
df = charts.load_df()
print(f"Loaded {len(df):,} rows from {DATA_CSV.name}")
df.head()


Loaded 39,040 rows from courtlistener.csv


,dataset,case,judge,court,court_id,date_of_decision,cite_count,court_level,judge_clean
0,King County (Wash. Ct. App.),"State Of Washington, V. Justin R. Smith",NaN,Court of Appeals of Washington,washctapp,2025-07-21,0,State: Wash. Court of Appeals (King County pull),NaN
1,King County (Wash. Ct. App.),"Chen Wang And Liyin Xue, V. Dongmei Huang",NaN,Court of Appeals of Washington,washctapp,2025-08-25,0,State: Wash. Court of Appeals (King County pull),NaN
2,King County (Wash. Ct. App.),In the Matter of the Detention of C.E.,NaN,Court of Appeals of Washington,washctapp,2025-09-09,0,State: Wash. Court of Appeals (King County pull),NaN
3,King County (Wash. Ct. App.),"Erwin Chappel, Respondent/cr-appellants V. Dou...",NaN,Court of Appeals of Washington,washctapp,2025-09-15,0,State: Wash. Court of Appeals (King County pull),NaN
4,King County (Wash. Ct. App.),"Alaska Airlines, V. Hillary Spanjer",NaN,Court of Appeals of Washington,washctapp,2025-09-15,0,State: Wash. Court of Appeals (King County pull),NaN


In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 39040 entries, 0 to 39039
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   dataset           39040 non-null  str           
 1   case              39040 non-null  str           
 2   judge             2888 non-null   str           
 3   court             39040 non-null  str           
 4   court_id          39040 non-null  str           
 5   date_of_decision  39040 non-null  datetime64[us]
 6   cite_count        39040 non-null  int64         
 7   court_level       39040 non-null  str           
 8   judge_clean       2888 non-null   str           
dtypes: datetime64[us](1), int64(1), str(7)
memory usage: 2.7 MB


In [4]:
df.describe(include="all")


,dataset,case,judge,court,court_id,date_of_decision,cite_count,court_level,judge_clean
count,39040,39040,2888,39040,39040,39040,39040.000000,39040,2888
unique,2,40,10,2,2,NaN,NaN,2,10
top,King County (Wash. Ct. App.),"State Of Washington, V. Justin R. Smith",Settle,Court of Appeals of Washington,washctapp,NaN,NaN,State: Wash. Court of Appeals (King County pull),Settle
freq,36000,1800,608,36000,36000,NaN,NaN,36000,608
mean,NaN,NaN,NaN,NaN,NaN,2025-02-05 20:57:56.065573,0.151844,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,2018-10-31 00:00:00,0.000000,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,2025-06-26 00:00:00,0.000000,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,2025-08-05 00:00:00,0.000000,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,2025-09-02 00:00:00,0.000000,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN,2025-09-22 00:00:00,7.000000,NaN,NaN


In [5]:
missing = df.isnull().sum()
print(missing)
print("judge_clean missing:", df["judge_clean"].isna().sum())


dataset                 0
case                    0
judge               36152
court                   0
court_id                0
date_of_decision        0
cite_count              0
court_level             0
judge_clean         36152
dtype: int64
judge_clean missing: 36152


### Data profile interpretation

**Rows and columns:** **39,040 rows × 9 columns** after loading (`dataset`, `case`, `judge`, `court`, `court_id`, `date_of_decision`, `cite_count`, plus derived `court_level` and `judge_clean`).

**What each column represents:**

| Column | Meaning |
|--------|---------|
| `dataset` | Source label for the CourtListener pull |
| `case` | Opinion / case title (caption) |
| `judge` | Authoring judge when present |
| `court` / `court_id` | Court name and short code (`wawd`, `washctapp`) |
| `date_of_decision` | Decision date |
| `cite_count` | Incoming citation clusters in this extract |

| Operation | One-sentence interpretation |
|-----------|----------------------------|
| **`head()`** | Each row is one opinion with a long case title, court metadata, a decision date, and a numeric cite count. |
| **`info()`** | Seven columns and ~39k rows; dates parse cleanly; judge is null on most rows because the state bulk dominates the file. |
| **`describe()`** | Overall mean cite is ~0.15 because ~36k state rows are zero; federal rows average ~1.95 cites in this extract. |
| **`isnull().sum()`** | **36,152** missing `judge_clean` values; `cite_count` has no nulls but is **uniformly 0** for `washctapp`. |

**Data quality issues:** Missing judges on state rows; misleading global means if state and federal rows are pooled; duplicate rows per case caption in the federal slice.

**Columns this analysis focuses on:** `cite_count` (outcome), `court_id` / `court_level` (population), `date_of_decision` (time), `case` (title-level ranking), and `judge_clean` (federal-only judge comparisons).


---

## Section 3 — Analysis

Each research question has: (1) the question stated below, (2) a **pandas** code cell, (3) a Plotly chart, and (4) an **Interpretation** cell. Section 4 summarizes visualization choices; Section 5 holds a short conclusions summary in this notebook; full **Section 4–5** (findings + process) and competency claims are in **`mp1.md`**.

### Interactive dashboard ([Dash gallery](https://dash.gallery/Portal/) style)

For **better exploration** than static notebook cells — zoom the line and treemap charts, orbit the 3D jurisdiction view, and use Mantine sliders on charts (b) and (c) — run the local Dash app (same data, polished theme):

```bash
cd MiniProject1
python mp1_dash.py
```

Then open **http://127.0.0.1:8050/**. You can also launch from the notebook cell below (opens in your browser).


In [6]:
# Optional — launch interactive Dash dashboard (stop kernel or Ctrl+C in terminal to end server)
import mp1_dash

app = mp1_dash.create_app(df)
print("Starting Dash at http://127.0.0.1:8050/ — use the browser tab that opens.")
app.run(debug=False, host="127.0.0.1", port=8050, jupyter_mode="external")

Starting Dash at http://127.0.0.1:8050/ — use the browser tab that opens.
Dash app running on http://127.0.0.1:8050/


### Question 1 — Citations by court level over time

Does citation frequency vary by court level? (Judge-level view is limited to the federal slice in this CSV.)  
**Chart:** dual line series (monthly mean `cite_count`) — not a bar chart.


In [7]:
# Question 1 — pandas: mean cite_count by month and court
q1 = df.dropna(subset=["date_of_decision"]).copy()
q1["year_month"] = q1["date_of_decision"].dt.to_period("M")
q1_monthly = (
    q1.groupby(["year_month", "court_id"], as_index=False)
    .agg(mean_cite=("cite_count", "mean"), n_opinions=("case", "count"))
    .sort_values(["year_month", "court_id"])
)
print("Latest monthly buckets:")
print(q1_monthly.tail(8).to_string(index=False))

Latest monthly buckets:
year_month  court_id  mean_cite  n_opinions
   2019-05      wawd   2.666667         912
   2019-07      wawd   2.000000         304
   2019-08      wawd   0.000000         152
   2025-04 washctapp   0.000000        1800
   2025-06 washctapp   0.000000        5400
   2025-07 washctapp   0.000000        7200
   2025-08 washctapp   0.000000       10800
   2025-09 washctapp   0.000000       10800


In [8]:
fig_q1 = charts.fig_q1_scatter3d_monthly(df)
fig_q1.show(config=charts.GRAPH_CONFIG)


**Interpretation:** The state appellate slice stays at **mean cite = 0** over time; the federal district shows **non-zero** monthly means. Court level must be named before anyone infers how “cited” local opinions are. I would investigate whether another API field carries state cites.


### Question 2 — Most cited judgments (W.D. Wash.)

Top case titles in the federal district by max `cite_count` in this pull.


In [9]:
# Question 2 — pandas: top cited case titles in W.D. Wash. (wawd)
q2 = (
    df[df["court_id"] == "wawd"]
    .groupby("case", as_index=False)
    .agg(max_cite=("cite_count", "max"), opinion_rows=("cite_count", "count"))
    .sort_values("max_cite", ascending=False)
    .head(12)
)
print(q2.to_string(index=False))

                                             case  max_cite  opinion_rows
                            O'Connor v. Berryhill         7           152
                   Hoober v. Movement Mortg., LLC         5           152
                Pengbo Xiao v. Feast Buffet, Inc.         5           152
                             Galvez v. Cuccinelli         4           152
       Mass. Bay Ins. Co. v. Walflor Indus., Inc.         4           152
Animal Legal Def. Fund v. Olympic Game Farm, Inc.         3           152
     Gallupe v. Sedgwick Claims Mgmt. Servs. Inc.         3           152
                  Beane v. RPW Legal Servs., PLLC         2           152
                  State v. Franciscan Health Sys.         1           152
                     Perfect Co. v. Adaptics Ltd.         1           152
                               Mitchell v. Atkins         1           152
             Ken M. ex rel. Berry M. v. Berryhill         1           152


In [10]:
fig_q2 = charts.fig_q2_scatter3d_jurisdiction(df, court_id="wawd", top_n=45, rank_mode="max")
fig_q2.show(config=charts.GRAPH_CONFIG)


**Interpretation:** Citation strength in `wawd` is **concentrated at the top ranks**; many rows repeat the same case caption (depth on the z-axis). Useful as a **starting list**, not a full citator replacement.


### Question 3 — Top authority titles (dataset-wide)

Summed `cite_count` by case title; title states the mean cite per row.  
**Chart:** treemap (tile area = total cite mass) — not a bar chart.


In [11]:
# Question 3 — pandas: dataset-wide mean and top titles by summed cite_count
print(f"Mean cite_count per opinion row: {df['cite_count'].mean():.4f}")
q3 = (
    df.groupby("case", as_index=False)
    .agg(total_cite=("cite_count", "sum"), opinion_rows=("cite_count", "count"))
    .sort_values("total_cite", ascending=False)
    .head(12)
)
print(q3.to_string(index=False))

Mean cite_count per opinion row: 0.1518
                                             case  total_cite  opinion_rows
                            O'Connor v. Berryhill        1064           152
                   Hoober v. Movement Mortg., LLC         760           152
                Pengbo Xiao v. Feast Buffet, Inc.         760           152
       Mass. Bay Ins. Co. v. Walflor Indus., Inc.         608           152
                             Galvez v. Cuccinelli         608           152
Animal Legal Def. Fund v. Olympic Game Farm, Inc.         456           152
     Gallupe v. Sedgwick Claims Mgmt. Servs. Inc.         456           152
                  Beane v. RPW Legal Servs., PLLC         304           152
                               Mitchell v. Atkins         152           152
                  State v. Franciscan Health Sys.         152           152
                     Perfect Co. v. Adaptics Ltd.         152           152
             Ken M. ex rel. Berry M. v. Berryhil

In [12]:
fig_q3 = charts.fig_q3_top_authorities(df, top_n=45, min_opinion_rows=1)
fig_q3.show(config=charts.GRAPH_CONFIG)


**Interpretation:** The **global mean** is low because of zero-heavy state rows, but **total cite mass** still piles onto a federal-heavy long tail of titles — “average row” and “top authority” are different questions.


In [13]:
# Save static chart images to images/ (also committed for reviewers)
try:
    written = charts.export_chart_images(fig_q1, fig_q2, fig_q3, out_dir=IMAGES_DIR)
    for p in written:
        print(p.relative_to(PROJECT_DIR))
except ImportError as e:
    print("Kaleido export skipped:", e)
    print("Using committed JPGs in images/")
    for p in sorted(IMAGES_DIR.glob("mp1a_chart*.jpg")):
        print(p.relative_to(PROJECT_DIR))


wrote /Users/rnr6/Documents/HCDE/hcde530/MiniProject1/images/mp1a_chart1_court_level_cites_3d.jpg
wrote /Users/rnr6/Documents/HCDE/hcde530/MiniProject1/images/mp1a_chart2_top_cited_cases_wd_wash.jpg
wrote /Users/rnr6/Documents/HCDE/hcde530/MiniProject1/images/mp1a_chart3_top_authority_titles.jpg
images/mp1a_chart1_court_level_cites_3d.jpg
images/mp1a_chart2_top_cited_cases_wd_wash.jpg
images/mp1a_chart3_top_authority_titles.jpg


### Static chart files (also in `images/`)

Snapshots below match the Plotly figures in this notebook (regenerate with `python mp1_charts.py`).

![Chart 1 — court level over time (line chart)](images/mp1a_chart1_court_level_over_time.jpg)

![Chart 2 — top cited cases W.D. Wash. (3D scatter)](images/mp1a_chart2_top_cited_cases_wd_wash.jpg)

![Chart 3 — top authority titles (Turbo treemap)](images/mp1a_chart3_top_authority_titles.jpg)


---

## Section 4 — Visualization

Three Plotly charts (Section 3) support the findings. Each has a **finding-style title**, labeled axes, and a chart type matched to the question.

| Chart | Type | Why this chart type | What the reader should take away |
|-------|------|---------------------|----------------------------------|
| **Q1 — court level over time** | Line chart with markers (month × mean cite, colored by court) | **Time series** makes the flat state line vs rising federal line easy to compare without rotating a 3D scene | Federal monthly means rise above zero; state stays at zero in this pull — label court level before comparing “how cited” opinions are |
| **Q2 — 3D rank × cite × depth** | 3D scatter | Surfaces **head of the distribution** and duplicate-row depth per title | Citation strength in W.D. Wash. is concentrated in a few titles; use as a starter list, not a full citator |
| **Q3 — authority titles** | Treemap (area = summed cite_count; **Turbo** color by total cites) | **Part-to-whole** view: large, warm-colored tiles hold the most citation mass | A small set of titles dominates total cite mass; global mean (~0.15) and treemap “mass” answer different questions |

Static exports for reviewers are in `images/mp1a_chart*.jpg` (Kaleido). For live exploration (rotate 3D, sliders), run `python mp1_dash.py` and open http://127.0.0.1:8050/.

---

## Section 5 — Conclusions

**Summary of findings (3–5 sentences):** In this CourtListener extract, **citation signal lives almost entirely in the federal W.D. Wash. slice** — roughly 36,000 state appellate rows show `cite_count = 0`, so any dataset-wide average is misleading unless courts are labeled. Within `wawd`, a steep head of case titles carries the highest max cites, while summed cites across the full file highlight a federal-heavy long tail of “authority” titles. The most important takeaway for practice is to **separate populations before comparing cite metrics**, then use charts as exploration aids backed by pandas tables, not as a complete citator.

**Most important finding:** Court level dominates whether `cite_count` is informative in this pull; federal monthly means vary, state means do not.

**What surprised me:** The state bulk was so large (~92% of rows) and so uniformly zero on `cite_count` that my first instinct to rank “most cited Washington opinions” from the whole CSV would have been wrong without profiling.

**What I would investigate next:** Whether CourtListener exposes a different cite field for `washctapp`; federal-only judge rankings with deduplicated captions; and whether duplicate rows per `case` title inflate Q2/Q3 bars.

**Limitations:** This is one API export, not a complete citator; `cite_count` meaning may differ by court; missing judges block judge-level claims on state rows; duplicate captions in the pull affect title-level aggregates; and static JPGs freeze one camera angle on 3D views.

### By question (brief)

**Question 1:** Federal rows carry the only meaningful `cite_count` variation in this extract; restrict time-series views to labeled court slices.

**Question 2:** Within W.D. Wash., citation strength concentrates at the top ranks — verify in a dedicated legal database before citing in briefs.

**Question 3:** Global mean (~0.15) and top summed titles answer different questions; report both with population notes.


---

## Section 6 — Process (optional)

Full **Section 5 — Process** narrative and expanded **Section 4 — Conclusions** (plain language per question) are in **`mp1.md`**. Competency claims (C3, C5, C6, C7) are also in that file.

**Standalone folder:** I copied `courtlistener.csv` and chart logic into `MiniProject1/` so graders need only this directory.

**Weeks 4–6:** API stacking (Week 5), then Plotly/Dash (Week 6). The important pivot was discovering **36k `washctapp` rows with cite_count = 0** — I stopped implying one aggregate speaks for all courts.

**Tooling:** Cursor helped with Dash layout; invalid Mantine props still failed at runtime until fixed. Kaleido exports required a venv/`pip install kaleido`; this notebook’s setup cell installs dependencies for a clean **Restart & Run All**.

**AI:** Suggested extra judge charts; I dropped them so each figure maps to one MP1a question. Conclusions here reflect the corrected story after the slice-quality check.
